# 01 - Service A (API Calls)

This notebook implements the API-backed service and transforms raw API output into user-friendly text.

## Objectives
- Select one public API backend.
- Validate user inputs.
- Transform API JSON into non-verbatim natural-language responses.
- Add error handling and fallback messages.

## Design Notes (Service A)

### API choice
- This implementation uses **Open-Meteo** because it is free, public, and does not require a secret key for current weather lookups.
- A geocoding step converts a city name into latitude/longitude before querying weather data.

### Why this satisfies assignment requirements
- The backend is a real public API service.
- The returned JSON is transformed into a human-friendly summary (not returned verbatim).
- Invalid input and API lookup failures are handled with safe error messages.

# Load Secrets

In [19]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [20]:
# Environment + model setup (aligned with assignment_1.ipynb)
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

if "deploying-ai-env" not in sys.executable.replace("\\", "/"):
    print("Warning: Activate virtual env 'deploying-ai-env' before running this notebook.")


client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
MODEL_NAME = "gpt-4o"

# user_prompt_response = client.responses.create(
#     model = 'gpt-4o',
#     input = "test",
# )
# print(user_prompt_response.output_text.strip())

# Notebook-specific imports
import requests
from typing import Dict, Any

In [ ]:
# Service A implementation: Public API + transformed response
from typing import Optional

# Open-Meteo endpoints (free, no API key needed)
GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"

# Human-friendly weather code mapping for current weather
WEATHER_CODE_MAP = {
    0: "clear sky",
    1: "mainly clear",
    2: "partly cloudy",
    3: "overcast",
    45: "fog",
    48: "depositing rime fog",
    51: "light drizzle",
    53: "moderate drizzle",
    55: "dense drizzle",
    61: "slight rain",
    63: "moderate rain",
    65: "heavy rain",
    71: "slight snow",
    73: "moderate snow",
    75: "heavy snow",
    80: "rain showers",
    81: "moderate rain showers",
    82: "violent rain showers",
    95: "thunderstorm",
}

def _validate_city_query(user_query: str) -> str:
    """Validate and normalize a user city query."""
    city = (user_query or "").strip()
    if not city:
        raise ValueError("Please provide a city name.")
    if len(city) < 2:
        raise ValueError("City name is too short.")
    if len(city) > 80:
        raise ValueError("City name is too long.")
    return city

def _geocode_city(city: str) -> Dict[str, Any]:
    """Convert city name into latitude/longitude using Open-Meteo geocoding."""
    response = requests.get(
        GEOCODING_URL,
        params={"name": city, "count": 1, "language": "en", "format": "json"},
        timeout=15,
    )
    response.raise_for_status()
    data = response.json()
    results = data.get("results") or []
    if not results:
        raise LookupError(f"I could not find a location match for '{city}'.")
    top = results[0]
    return {
        "name": top.get("name", city),
        "country": top.get("country", "Unknown"),
        "latitude": top["latitude"],
        "longitude": top["longitude"],
    }

def call_public_api(user_query: str) -> Dict[str, Any]:
    """Service A backend: fetch current weather for a city from a public API."""
    city = _validate_city_query(user_query)
    location = _geocode_city(city)

    response = requests.get(
        WEATHER_URL,
        params={
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "current_weather": True,
            "timezone": "auto",
        },
        timeout=15,
    )
    response.raise_for_status()
    weather_data = response.json()
    current = weather_data.get("current_weather")
    if not current:
        raise RuntimeError("Weather service returned an unexpected response.")

    return {
        "service": "open-meteo",
        "location": location,
        "current_weather": current,
    }


In [23]:

def transform_api_payload(payload: Dict[str, Any], use_llm_rewrite: bool = False) -> str:
    """Transform structured weather payload into natural-language (non-verbatim) response."""
    location = payload["location"]
    current = payload["current_weather"]

    code = current.get("weathercode")
    weather_desc = WEATHER_CODE_MAP.get(code, "unclassified weather")
    temp = current.get("temperature")
    wind = current.get("windspeed")
    observed_at = current.get("time", "latest update")

    base_text = (
        f"In {location['name']}, {location['country']}, it is currently {weather_desc} "
        f"with a temperature of {temp}°C and wind speed near {wind} km/h "
        f"(observed at {observed_at})."
    )

    if not use_llm_rewrite:
        return base_text

    # Optional: rewrite the response with the course model for a more conversational style.
    rewrite_prompt = f"""
        Rewrite the following weather update into a concise, friendly and funny one-paragraph message.
        Keep the facts unchanged and do not add new data but you can add a joke if you want.

        Weather update: {base_text}
        """.strip()
    import traceback
    try:
        llm_response = client.responses.create(
            model=MODEL_NAME,
            input=[{"role": "user", "content": rewrite_prompt}],
            temperature=0.2,
        )

        # user_prompt_response = client.responses.create(
        #     model = 'gpt-4o',
        #     input = "HELLO",
        # )
        # print(f""" user prompt response: {user_prompt_response.output_text.strip()}""")
        # print(f""" response: {llm_response.output_text.strip()}""")

        return llm_response.output_text.strip()
    except Exception:
        # Safe fallback to deterministic text if model rewrite fails.
        print("LLM rewrite failed, falling back to base text.")
        #print the exception for debugging purposes 
        
        traceback.print_exc()
        return base_text

In [ ]:
# Quick demo tests for Service A
sample_queries = [
    "Toronto",
    "New York",
    "NonExistentCity123",  # Should trigger geocoding error
    "", # Too short, should trigger validation error
    "Q"  # Too short, should trigger validation error
    ]

for query in sample_queries:
    print(f"\n--- Query: {query!r}")
    try:
        payload = call_public_api(query)
        response_text = transform_api_payload(payload, use_llm_rewrite=True)
        print(response_text)
    except Exception as exc:
        print(f"Handled error: {exc}")


--- Query: 'Toronto'
Hey Toronto! The sky's as clear as your New Year's resolutions, with a brisk 3.2°C keeping things cool. Winds are breezing by at 7.9 km/h—just enough to mess up your hair but not your day!

--- Query: 'New York'
Hey New Yorkers! The sky's as clear as your weekend plans, with a brisk 1.9°C to keep you on your toes. Winds are cruising at 6.4 km/h, so hold onto your hats—or your pizza slices!

--- Query: 'NonExistentCity123'
Handled error: I could not find a location match for 'NonExistentCity123'.

--- Query: ''
Handled error: Please provide a city name.

--- Query: 'QC'
Handled error: I could not find a location match for 'QC'.


## Notes
- Set `use_llm_rewrite=False` for deterministic behavior.
- Set `use_llm_rewrite=True` for model-polished phrasing.
- [x] API response is not returned verbatim.
- [x] Input validation works for invalid queries (cities).
- [x] API failures return safe and helpful messages.